# PlantNet Resnet 18 Finetuning

Finetuning Resnet 18 on the plantnet dataset.

In [20]:
import torch
import torchvision.models as models

batch_size = 256
epochs = 1
cores = 10

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

device = torch.device("cuda:0")

# Load the pre-trained ResNet-18 model
model = models.resnet18(pretrained=True)

num_classes = 1081
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

model.to(device)


CUDA Available: True
GPU Name: NVIDIA GeForce RTX 3060 Ti


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## Loading and Normalization


In [22]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader

# recommended normalizations per https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageFolder('../dataset/train', transform=transform)
val_dataset = ImageFolder('../dataset/val', transform=transform)
test_dataset = ImageFolder('../dataset/test', transform=transform)

# prevent learning on the order of the data
train_loader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=cores)
val_loader = DataLoader(val_dataset, batch_size, shuffle=False, num_workers=cores)
test_loader = DataLoader(test_dataset, batch_size, shuffle=False, num_workers=cores)

## Loss Function/Optimizer

In [23]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)

## Training Function


In [24]:
train_losses = []
test_losses = []

train_correct = []
test_correct = []

def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    for e in range(num_epochs):
        running_loss = 0
        running_corr = 0
        for b, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            predicted_labels = model(inputs)
            loss = criterion(predicted_labels, labels)

            # predicted labels
            predicted = torch.max(predicted_labels, 1)[1]

            # sum the number of correct predictions
            batch_corr = (predicted == labels).sum()

            # track loss and correct predictions
            running_corr += batch_corr
            running_loss += loss.item() * inputs.size(0)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            print(f"batch {b}, epoch {e}")

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corr.float() / len(train_loader.dataset)

        train_losses.append(train_loss)
        train_correct.append(train_acc)

        running_loss = 0
        running_corr = 0
        with torch.no_grad():
            for b, (X_test, Y_test) in enumerate(val_loader):
                inputs, labels = X_test.to(device), Y_test.to(device)
                predicted_labels = model(inputs)
                predicted = torch.max(predicted_labels, 1)[1]
                test_corr = (predicted == labels).sum()

                loss = criterion(predicted_labels, labels)

                running_corr += test_corr
                running_loss += loss.item() * inputs.size(0)
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corr.float() / len(val_loader.dataset)

        print(f"Epoch: {e+1}/{num_epochs}, train_loss: {train_loss}, train_acc: {train_acc}, val_loss: {val_loss}, val_acc: {val_acc}")




##  Fine Tuning

In [25]:
train(model, train_loader, val_loader, criterion, optimizer, epochs)

batch 0, epoch 0
batch 1, epoch 0
batch 2, epoch 0
batch 3, epoch 0
batch 4, epoch 0
batch 5, epoch 0
batch 6, epoch 0
batch 7, epoch 0
batch 8, epoch 0
batch 9, epoch 0
batch 10, epoch 0
batch 11, epoch 0
batch 12, epoch 0
batch 13, epoch 0
batch 14, epoch 0
batch 15, epoch 0
batch 16, epoch 0
batch 17, epoch 0
batch 18, epoch 0
batch 19, epoch 0
batch 20, epoch 0
batch 21, epoch 0
batch 22, epoch 0
batch 23, epoch 0
batch 24, epoch 0
batch 25, epoch 0
batch 26, epoch 0
batch 27, epoch 0
batch 28, epoch 0
batch 29, epoch 0
batch 30, epoch 0
batch 31, epoch 0
batch 32, epoch 0
batch 33, epoch 0
batch 34, epoch 0
batch 35, epoch 0
batch 36, epoch 0
batch 37, epoch 0
batch 38, epoch 0
batch 39, epoch 0
batch 40, epoch 0
batch 41, epoch 0
batch 42, epoch 0
batch 43, epoch 0
batch 44, epoch 0
batch 45, epoch 0
batch 46, epoch 0


KeyboardInterrupt: 

## Eval


In [10]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(model, test_loader, device):
    correct_pred = {classname: 0 for classname in test_loader.dataset.classes}
    total_pred = {classname: 0 for classname in test_loader.dataset.classes}

    model.eval()

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            for label, prediction in zip(labels, preds):
                classname = test_loader.dataset.classes[label]
                if label == prediction:
                    correct_pred[classname] += 1
                total_pred[classname] += 1

    accuracy_per_class = {classname: correct_pred[classname] / total_pred[classname]}

    overall_acc = accuracy_score(all_labels, all_preds)

    for classname, acc in accuracy_per_class.items():
        print(f"{classname}: {acc:.4f}%")

    print()
    print(f"overall acc: {overall_acc:.4f}")

evaluate_model(model, test_loader, device)


RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)